# Spanish tutor: the data pipeline, step by step

This notebook walks through how the tutor's data is loaded, processed, vectorized and
stored, and why each design decision was made. It runs the project's own code
(`src/spanish_tutor/`), so what you see here is exactly what the tutor uses.

```
SUBTLEX-ESP ─┐                                  ┌─> word bank (SQLite)
Tatoeba ─────┼─> parse ─> lemmatize ─> filter ──┤
Wiktionary ──┘           (lemma, pos)           └─> embed ─> vector store (Chroma)
```

| Section | What it covers |
|---|---|
| 1. Data sets | The three sources, with links, licenses and raw samples |
| 2. Processing | How text becomes `(lemma, pos)` pairs, word frequencies, and the word bank |
| 3. Vectorization | The embedding model, one sentence's vector, and a 2D PCA check |
| 4. Storage | How sentences and vectors are stored in Chroma |
| 5. Lookups | Searching the store against the word bank, and looking up words in SQLite |

**Before running:** the pipeline commands in the README must have been run, so that
`data/raw/`, `data/processed/word_bank.db` and `data/processed/chroma/` exist. Use the
project's `.venv` as the kernel.

**Licensing note:** the outputs below include excerpts of SUBTLEX-ESP (CC BY-NC-SA),
Tatoeba (CC BY 2.0 FR) and Wiktionary (CC BY-SA), plus your personal word bank. Clear
outputs before committing this notebook to the public repo.

In [ ]:
import itertools
import json
import warnings

# sentence-transformers imports tqdm, which warns that notebook progress widgets
# aren't installed. Harmless: we don't use progress bars here.
warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
from sklearn.decomposition import PCA

from spanish_tutor import db
from spanish_tutor.config import CHROMA_DIR, DB_PATH
from spanish_tutor.ingest import subtlex
from spanish_tutor.ingest.download import RAW_DIR, WIKTIONARY_FILE
from spanish_tutor.ingest.tatoeba import ANALYZED_PATH, read_analyzed, read_tsv
from spanish_tutor.ingest.wiktionary import Wiktionary

pd.set_option("display.max_colwidth", 90)
print("raw data:      ", RAW_DIR)
print("word bank:     ", DB_PATH)
print("vector store:  ", CHROMA_DIR)

## 1. Data sets

Three open data sources feed the pipeline. Each answers a different question:

| Source | Question it answers | License |
|---|---|---|
| SUBTLEX-ESP | How common is each word in everyday speech? | CC BY-NC-SA 4.0 |
| Tatoeba | How is each word actually used, in real sentences with translations? | CC BY 2.0 FR |
| Wiktionary | Is this a real word, and what does it mean? | CC BY-SA 4.0 |

**Design decision: the repository ships code, never data.** Each source is downloaded
at build time into `data/raw/` (gitignored), because the licenses attach conditions to
redistribution (non-commercial, share-alike, per-author credit).

### 1.1 SUBTLEX-ESP: word frequencies from film subtitles

- **What:** frequency counts for 94,261 Spanish word forms, from about 40 million words
  of film and TV subtitles.
- **Why subtitles:** subtitles approximate everyday spoken language better than books or
  news. The SUBTLEX authors found that subtitle frequencies predict how quickly people
  recognize words better than older written-text frequency lists did.
- **Source:** Cuetos, F., Glez-Nosti, M., Barbón, A., & Brysbaert, M. (2011).
  *SUBTLEX-ESP: Spanish word frequencies based on film subtitles.* Psicológica, 32, 133–143.
  Project page: [osf.io/xp6sz](https://osf.io/xp6sz/), file: [SUBTLEX-ESP.xlsx](https://osf.io/download/fxt57/).

The raw file has an unusual layout: one sheet with the list split into **three
side-by-side column blocks**, sorted alphabetically. Here are its first rows:

In [ ]:
workbook = openpyxl.load_workbook(RAW_DIR / "SUBTLEX-ESP.xlsx", read_only=True)
raw_rows = list(itertools.islice(workbook.worksheets[0].iter_rows(values_only=True), 6))
workbook.close()
pd.DataFrame(
    raw_rows[1:], columns=[f"{c} ({i // 5 + 1})" if c else "" for i, c in enumerate(raw_rows[0])]
)

`subtlex.load_counts()` reads all three blocks, normalizes each form (Unicode NFC,
lowercase) and sums the 76 words that appear twice. The result maps each **word form**
to its count. It has no lemmas and no parts of speech: *hablo*, *hablas* and *habló* are
separate rows. Section 2 deals with that.

In [ ]:
form_counts = subtlex.load_counts(RAW_DIR / "SUBTLEX-ESP.xlsx")
total = sum(form_counts.values())
print(f"{len(form_counts):,} word forms, {total:,} subtitle tokens\n")

top = pd.DataFrame(form_counts.most_common(15), columns=["form", "count"])
top["per million"] = (top["count"] / total * 1e6).round(0).astype(int)
top

**Design decision: accents are kept.** Spanish uses accents to tell words apart, so
normalization never strips them:

In [ ]:
pd.DataFrame(
    [(f, form_counts[f]) for f in ["si", "sí", "esta", "está", "el", "él", "que", "qué"]],
    columns=["form", "count"],
)

### 1.2 Tatoeba: real sentences with translations

- **What:** a community-built collection of sentences and their translations, made for
  language learners. It has 442,260 Spanish sentences, 261,405 of which have an English
  translation.
- **Why:** it supplies real usage. That serves three purposes: deciding how ambiguous
  forms are used, providing example sentences, and grounding retrieval.
- **Source:** [tatoeba.org/en/downloads](https://tatoeba.org/en/downloads). We use the
  weekly per-language exports:
  - [spa_sentences_detailed.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/spa/spa_sentences_detailed.tsv.bz2) (id, language, text, author, dates)
  - [eng_sentences.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/eng/eng_sentences.tsv.bz2) (id, language, text)
  - [spa-eng_links.tsv.bz2](https://downloads.tatoeba.org/exports/per_language/spa/spa-eng_links.tsv.bz2) (Spanish id → English id)
- **License:** CC BY 2.0 FR. Each sentence's author must be credited, so the author
  travels with every sentence through the whole pipeline.

The raw files are tab-separated with no header row, and use `\N` for "unknown":

In [ ]:
for name in ["spa_sentences_detailed.tsv.bz2", "eng_sentences.tsv.bz2", "spa-eng_links.tsv.bz2"]:
    print(f"--- {name}")
    for row in itertools.islice(read_tsv(RAW_DIR / name), 3):
        print("   ", row)

`tatoeba.load_sentences()` joins the three files: each Spanish sentence gets its
lowest-id English translation (if any) and its author. A sample of the result:

In [ ]:
sample = [s for s in itertools.islice(read_analyzed(ANALYZED_PATH), 2000, 2400) if s.en][:8]
pd.DataFrame(
    [(s.id, s.es, s.en, s.author) for s in sample], columns=["id", "Spanish", "English", "author"]
)

### 1.3 Wiktionary: is it a real word, and what does it mean?

- **What:** English Wiktionary's entries for Spanish words, in machine-readable form.
- **Source:** [kaikki.org](https://kaikki.org/dictionary/Spanish/), which extracts
  Wiktionary with *wiktextract* (Ylonen, 2022, LREC). We use the
  [Spanish-only extract](https://kaikki.org/dictionary/Spanish/kaikki.org-dictionary-Spanish.jsonl)
  (~95 MB compressed in transit) and fall back to the full multi-language dump if it
  disappears.
- **License:** CC BY-SA 4.0.

`ingest/download.py` reduces each entry to its word, part of speech and up to five
senses. **Design decision:** Wiktionary lists inflected forms as their own entries
(*pies* = "plural of *pie*"). Those are dropped for nouns, verbs and adjectives, so only
dictionary forms remain. They are *kept* for function words, because a learner has to
learn *ti* ("prepositional of *tú*") as a separate word.

In [ ]:
wikt_path = RAW_DIR / WIKTIONARY_FILE
entries = {}
with wikt_path.open(encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        if entry["word"] in {"bajo", "ti"}:
            entries.setdefault(entry["word"], []).append(entry)

for word, word_entries in entries.items():
    for entry in word_entries[:3]:
        print(f"{word} ({entry['pos']})")
        for sense in entry["senses"][:3]:
            print(f"    - {sense['gloss']}   tags={sense['tags']}")

The `Wiktionary` class indexes those entries by `(lemma, UD part of speech)`, the same
key the word bank uses. Two design decisions matter here:

- **Part-of-speech fallback for function words.** spaCy and Wiktionary sometimes disagree:
  spaCy tags *cómo* as a pronoun, Wiktionary calls it an adverb. For function words only,
  a compatible part of speech counts as a match. The word keeps spaCy's tag, since that's
  what content indexing will produce.
- **Following "alternative spelling" entries.** *ese* is listed only as "alternative
  spelling of *ése*", so the lookup follows it to the real definition.

In [ ]:
wiktionary = Wiktionary(wikt_path)
pd.DataFrame(
    [
        (lemma, pos, wiktionary.definition(lemma, pos))
        for lemma, pos in [
            ("bajo", "ADJ"),
            ("bajo", "ADP"),
            ("cómo", "PRON"),
            ("ese", "PRON"),
            ("ti", "PRON"),
        ]
    ],
    columns=["lemma", "pos", "definition"],
)

## 2. Processing: from text to `(lemma, pos)`

The word bank stores **words**, not word forms. If you know *hablar*, you can read
*hablamos* and *habló*. So all text, whether in subtitles, sentences or (later) songs and
books, is reduced to `(lemma, part of speech)` pairs.

**Design decision: a word is identified by its lemma *and* part of speech.** *bajo* the
adjective ("short"), *bajo* the preposition ("under") and *bajo* the noun ("bass") are
different words to learn, so they're tracked separately. The cost is that tagging
mistakes create spurious pairs. We accepted that cost.

**Design decision: one lemmatizer for everything.** `src/spanish_tutor/lexicon.py` is the
only code that turns text into `(lemma, pos)`. The word bank, the seed pipeline, the
vector store and the future content index all use it. If two parts of the system
lemmatized differently, they would disagree about whether a word is new.

### 2.1 What spaCy produces

We use [spaCy's `es_core_news_md`](https://spacy.io/models/es#es_core_news_md), trained
on the AnCora Spanish treebank. Its lemmatizer is rule-based: it uses each token's
predicted part of speech and morphology (tense, person, number, ...) to find the
dictionary form. Here is its raw output for one sentence:

In [ ]:
from spanish_tutor.lexicon import analyze, load_nlp, token_analyses, vocabulary

sentence = "Me lo dijo ella: no vengo del mercado porque hay mucho trabajo."
doc = load_nlp()(sentence)
raw = pd.DataFrame(
    [(t.text, t.lemma_, t.pos_, str(t.morph)) for t in doc],
    columns=["token", "spaCy lemma", "spaCy POS", "morphology"],
)
raw

Several of those raw results are wrong *for our purpose*, so `lexicon.normalize()`
applies these rules:

| Rule | Example | Why |
|---|---|---|
| Unicode NFC + lowercase, **accents kept** | *Sí* → *sí* (≠ *si*) | Accents distinguish words |
| `AUX` → `VERB` | *hay* (haber, AUX) → haber VERB | spaCy tags copular *ser/estar* and auxiliary *haber* as AUX; without folding, each verb would be two words |
| Contractions expanded | *del* → *de* ADP + *el* DET | spaCy leaves *del*/*al* whole |
| **Personal pronouns keep their own form** | *me* → *me* (not *yo*) | spaCy lemmatizes *me, nos, conmigo* to *yo* and *se, lo, le, ella* to *él*. Knowing *yo* doesn't mean knowing *conmigo* |
| Clitic verbs reduced to the verb | *dar yo él* → *dar* | spaCy joins attached pronouns into the lemma |
| Non-vocabulary dropped | names, punctuation, digits | Not words to learn |

The same sentence after normalization. Changed rows are marked:

In [ ]:
normalized = [", ".join(f"{l}|{p}" for l, p in token_analyses(t)) or "(dropped)" for t in doc]
raw["normalized (lemma|POS)"] = normalized
raw["changed"] = ["✓" if n != f"{t.lemma_}|{t.pos_}" else "" for t, n in zip(doc, normalized)]
raw[["token", "spaCy lemma", "spaCy POS", "normalized (lemma|POS)", "changed"]]

`analyze()` runs this over many texts at once (using spaCy's batched `nlp.pipe`), and
`vocabulary()` reduces a sentence to its set of distinct words:

In [ ]:
tokens = next(analyze([sentence]))
print("per token:", tokens[:6], "...")
print("\nvocabulary:", sorted(vocabulary(tokens)))

Lemmatizing all 442k Tatoeba sentences takes about 15 minutes, so
`python -m spanish_tutor.ingest.tatoeba` does it once and caches the result in
`data/processed/tatoeba_analyzed.jsonl`. One cached record:

In [ ]:
with ANALYZED_PATH.open(encoding="utf-8") as f:
    records = (json.loads(line) for line in itertools.islice(f, 2050, None))
    record = next(r for r in records if r["en"])  # the first translated one

for key in ["id", "es", "en", "author"]:
    print(f"{key + ':':8}{record[key]}")
print("tokens: (surface form, [(lemma, pos), ...])")
for form, analyses in record["tokens"]:
    print(f"        {form!r:14} -> {analyses}")

### 2.2 From word-form counts to word frequencies

SUBTLEX-ESP counts forms; the word bank needs words. A form like *bajo* can belong to
several `(lemma, pos)` pairs, and the subtitles don't say which one was meant.

**Design decision: split each form's count by how that form is used in real
sentences.** For every form *f*, Tatoeba tells us how often spaCy assigned it each
analysis. The SUBTLEX count is divided in the same proportions:

$$\text{freq}(\ell, p) = \sum_{f} \text{count}_{\text{SUBTLEX}}(f) \cdot \frac{n_{\text{Tatoeba}}(f \to (\ell, p))}{n_{\text{Tatoeba}}(f)}$$

A contraction counts fully toward both of its words (*del* is always both *de* and *el*).

In [ ]:
from spanish_tutor.seed import form_analysis_counts, lemma_frequencies

occurrences, with_analysis = form_analysis_counts(read_analyzed(ANALYZED_PATH))

rows = []
for form in ["bajo", "como", "del"]:
    for (lemma, pos), n in with_analysis[form].most_common(4):
        rows.append((form, form_counts[form], f"{lemma}|{pos}", n, n / occurrences[form]))
pd.DataFrame(
    rows, columns=["form", "SUBTLEX count", "analysis", "Tatoeba tokens", "share"]
).style.format({"share": "{:.0%}", "SUBTLEX count": "{:,}"})

In [ ]:
frequencies, dropped = lemma_frequencies(form_counts, occurrences, with_analysis)
print(f"{len(frequencies):,} distinct (lemma, pos) pairs")
print(f"{dropped / total:.1%} of subtitle tokens are forms never seen in Tatoeba (dropped)\n")
pd.DataFrame(
    [
        (rank, lemma, pos, round(freq))
        for rank, ((lemma, pos), freq) in enumerate(frequencies.most_common(15), 1)
    ],
    columns=["rank", "lemma", "pos", "estimated count"],
)

**Design decision: keep only pairs that Wiktionary knows.** This filter removes names,
English words that appear in subtitles, and tagging errors. Among the most frequent
pairs, almost everything rejected is a tagging error, like an inflected form treated as
a lemma (*crees*, *has*) or an invented lemma (*deberiar*):

In [ ]:
top_pairs = frequencies.most_common(3000)
rejected = [(f"{l}|{p}", round(c)) for (l, p), c in top_pairs if (l, p) not in wiktionary]
print(
    f"of the top {len(top_pairs):,} pairs, {len(rejected)} have no Wiktionary entry. The most frequent:"
)
pd.DataFrame(rejected[:12], columns=["pair", "estimated count"]).T

### 2.3 Storing vocabulary: the word bank

The top 1,500 accepted pairs went into a spreadsheet, and each one you knew was marked
`r` (recognize) or `p` (can produce). `python -m spanish_tutor.seed build` loaded the marks
into SQLite through a re-runnable SQL script. The schema (`sql/schema.sql`) has three
parts:

- **`lexemes`**: one row per `(lemma, pos)`, with English definition, example sentence and
  attribution (Tatoeba sentence id and author).
- **`word_events`**: an append-only log of every time a word is taught, seen, looked up or
  used. **Design decision:** the log is the source of truth, so scores can always be
  recalculated from full history.
- **`word_bank`**: one row per `(word, mode)`. Recognition and production are tracked
  separately, because the gap between understanding Spanish and producing it is what the
  project exists to close. **Design decision:** a trigger keeps this table current as
  events arrive. Computing it from the log on every read took 6.7 s per lookup at 1M
  events; the trigger table answers in 0.1 ms at 11M.

In [ ]:
conn = db.connect(DB_PATH)
pd.read_sql("SELECT mode, COUNT(*) AS words FROM word_bank GROUP BY mode", conn)

In [ ]:
# Words you recognize but can't yet produce, by part of speech: a ready-made practice list.
pd.read_sql(
    """
    SELECT l.pos, COUNT(*) AS words
    FROM word_bank AS r
    JOIN lexemes AS l ON l.lexeme_id = r.lexeme_id
    WHERE r.mode = 'recognition'
      AND NOT EXISTS (
          SELECT 1 FROM word_bank AS p
          WHERE p.lexeme_id = r.lexeme_id AND p.mode = 'production'
      )
    GROUP BY l.pos
    ORDER BY words DESC
    """,
    conn,
)

## 3. Vectorization

To find example sentences by **meaning** rather than by exact words, each sentence is
turned into a vector: a list of numbers positioned so that sentences with similar
meanings point in similar directions. Closeness is measured with **cosine similarity**
(1 = same direction, 0 = unrelated).

### 3.1 The model: `jina-embeddings-v2-base-es`

- **What:** a sentence-embedding model from Jina AI, trained on Spanish and English
  together. An English query finds Spanish sentences, and vice versa.
  Model card: [huggingface.co/jinaai/jina-embeddings-v2-base-es](https://huggingface.co/jinaai/jina-embeddings-v2-base-es).
- **Architecture:** JinaBERT, a 12-layer BERT variant that uses ALiBi attention biases
  instead of learned positions, which lets it read up to 8,192 tokens. Token vectors are
  mean-pooled into one 768-dimensional sentence vector.
- **License:** Apache-2.0.

Design decisions:

- **Why this model:** in a direct comparison with `multilingual-e5-base`, both ranked
  correctly, but jina separated related from unrelated sentences far more clearly:

  | Pair | jina | e5 |
  |---|---|---|
  | "Hace mucho calor hoy" ↔ "It is very hot today" | 0.895 | 0.924 |
  | "Hace mucho calor hoy" ↔ "Me gusta leer libros" | 0.038 | 0.841 |

  With jina, a similarity score means something on its own, so it can be used as a
  threshold.
- **The cost:** its architecture code is downloaded from Hugging Face
  (`trust_remote_code`), and that code doesn't run on transformers 5. So
  `transformers<5` and `sentence-transformers<6` are pinned, and the model and its code are
  pinned to exact commits (`src/spanish_tutor/embeddings.py`).
- **Normalized vectors:** every vector has length 1, so cosine similarity is simply the
  dot product.
- **CPU, via PyTorch:** about 130 sentences per second, so all 261k took about 35 minutes.

Loading the model (through LangChain's `HuggingFaceEmbeddings`) and printing its
structure confirms the pooling and the vector size:

In [ ]:
from spanish_tutor.embeddings import MODEL_ID, MODEL_REVISION, load_embeddings

embeddings = load_embeddings()
print(MODEL_ID, "@", MODEL_REVISION[:10])
print(embeddings._client)  # the underlying sentence-transformers model

### 3.2 One sentence and its vector

In [ ]:
anchor = "Tengo hambre, voy a comer algo."
vector = np.array(embeddings.embed_query(anchor))

print(f"sentence: {anchor!r}")
print(f"shape:    {vector.shape}   length (L2 norm): {np.linalg.norm(vector):.4f}")
print(f"range:    {vector.min():+.3f} … {vector.max():+.3f}\n")
pd.DataFrame(
    [vector[:10].round(4)], columns=[f"d{i}" for i in range(10)], index=["first 10 of 768"]
)

The individual numbers have no human meaning; only the *relative positions* of vectors
matter. To check that, we compare the anchor sentence with sentences that mean something
similar (in Spanish and English) and with sentences about unrelated topics.

### 3.3 Similar vs. different sentences

In [ ]:
examples = [
    # (sentence, group, language)
    (anchor, "anchor", "es"),
    ("Tengo mucha hambre.", "similar", "es"),
    ("¿Hay algo de comer en la cocina?", "similar", "es"),
    ("Quiero comer ahora mismo.", "similar", "es"),
    ("Vamos a almorzar juntos.", "similar", "es"),
    ("I'm hungry, I'm going to eat something.", "similar", "en"),
    ("I want to eat right now.", "similar", "en"),
    ("El tren sale a las ocho.", "different", "es"),
    ("Me gusta leer libros de historia.", "different", "es"),
    ("Mañana va a llover mucho.", "different", "es"),
    ("Mi hermano trabaja en un banco.", "different", "es"),
    ("The train leaves at eight.", "different", "en"),
]
frame = pd.DataFrame(examples, columns=["sentence", "group", "language"])
vectors = np.array(embeddings.embed_documents(frame["sentence"].tolist()))
frame["similarity to anchor"] = vectors @ vectors[0]  # unit vectors: dot product = cosine
frame.sort_values("similarity to anchor", ascending=False).style.format(
    {"similarity to anchor": "{:.3f}"}
)

### 3.4 Projecting to 2D with PCA

The vectors live in 768 dimensions. Principal component analysis finds the two directions
along which these twelve vectors vary most, and projects each vector onto them. If the
model works, the sentences about eating should land near the anchor, and the unrelated
sentences should land elsewhere.

**Caveat:** two dimensions keep only part of the variation (printed below), so distances
in the plot are approximate. Points that nearly coincide, like a sentence and its
translation, get their labels moved aside with thin leader lines. The cosine similarities above are the exact measure; the
plot is a visual check.

In [ ]:
pca = PCA(n_components=2)
points = pca.fit_transform(vectors)
frame["pc1"], frame["pc2"] = points[:, 0], points[:, 1]
explained = pca.explained_variance_ratio_
print(
    f"PC1 explains {explained[0]:.0%} and PC2 {explained[1]:.0%} of the variance ({explained.sum():.0%} together)"
)

# Palette: reference categorical slots 1 (blue) and 2 (orange), validated all-pairs;
# the anchor uses ink. Marker shape encodes language, so color is never the only cue.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
GROUP_STYLE = {
    "similar": {"color": "#2a78d6", "label": "Similar meaning"},
    "different": {"color": "#eb6834", "label": "Different topic"},
}
MARKER = {"es": "o", "en": "s"}

fig, ax = plt.subplots(figsize=(10, 6.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for group, style in GROUP_STYLE.items():
    for language, marker in MARKER.items():
        rows = frame[(frame.group == group) & (frame.language == language)]
        ax.scatter(
            rows.pc1,
            rows.pc2,
            s=90,
            marker=marker,
            color=style["color"],
            edgecolor=SURFACE,
            linewidth=2,
            zorder=3,
        )
a = frame.iloc[0]
ax.scatter(a.pc1, a.pc2, s=260, marker="*", color=INK, edgecolor=SURFACE, linewidth=2, zorder=4)

# Legend: color = meaning group, shape = language, star = anchor.
from matplotlib.lines import Line2D

handles = [
    Line2D([], [], marker="*", linestyle="", markersize=14, color=INK, label="Anchor sentence")
]
handles += [
    Line2D([], [], marker="o", linestyle="", markersize=9, color=s["color"], label=s["label"])
    for s in GROUP_STYLE.values()
]
handles += [
    Line2D([], [], marker=m, linestyle="", markersize=8, color=MUTED, label=lang)
    for lang, m in [("Spanish", "o"), ("English", "s")]
]
ax.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=False,
    fontsize=9,
    labelcolor=INK_2,
)

ax.set_xlabel(f"PC1 ({explained[0]:.0%} of variance)", color=MUTED, fontsize=9)
ax.set_ylabel(f"PC2 ({explained[1]:.0%} of variance)", color=MUTED, fontsize=9)
ax.set_title(
    "Sentences about eating cluster around the anchor, in both languages",
    loc="left",
    color=INK,
    fontsize=12,
    pad=12,
)
ax.grid(color=GRID, linewidth=0.6, zorder=0)
ax.tick_params(colors=MUTED, labelsize=8, length=0)
for side in ax.spines.values():
    side.set_visible(False)
# Extra room on the left: the anchor cluster sits near the left edge and needs label space.
span_x, span_y = np.ptp(points[:, 0]), np.ptp(points[:, 1])
ax.set_xlim(points[:, 0].min() - 0.55 * span_x, points[:, 0].max() + 0.4 * span_x)
ax.set_ylim(points[:, 1].min() - 0.2 * span_y, points[:, 1].max() + 0.2 * span_y)
plt.tight_layout()

# Label placement. Close points (a sentence and its translation) would stack their labels,
# so each label tries several positions and keeps the first that overlaps nothing already
# drawn: other labels, the markers, or the plot edge. Far positions get a thin leader line.
from matplotlib.text import Text
from matplotlib.transforms import Bbox

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
plot_area = ax.get_window_extent(renderer)
occupied = []
for x, y in zip(frame.pc1, frame.pc2):
    px, py = ax.transData.transform((x, y))
    occupied.append(Bbox.from_extents(px - 8, py - 8, px + 8, py + 8))

CANDIDATES = [(8, 3), (8, -13), (-8, 3), (-8, -13)]
CANDIDATES += [
    (dx, dy) for dist in (22, 38, 54, 70, 86, 102) for dy in (dist, -dist - 10) for dx in (16, -16)
]

for _, row in frame.iterrows():  # the anchor is first, so it gets the closest spot
    text = row.sentence if len(row.sentence) <= 34 else row.sentence[:32] + "…"
    style = {
        "fontsize": 9,
        "color": INK if row.group == "anchor" else INK_2,
        "fontweight": "bold" if row.group == "anchor" else "normal",
        "textcoords": "offset points",
        "zorder": 5,
    }
    for dx, dy in CANDIDATES:
        leader = (
            {"arrowstyle": "-", "color": MUTED, "lw": 0.6, "shrinkB": 5} if abs(dy) > 15 else None
        )
        label = ax.annotate(
            text,
            (row.pc1, row.pc2),
            xytext=(dx, dy),
            ha="left" if dx > 0 else "right",
            va="bottom",
            arrowprops=leader,
            **style,
        )
        # Measure the text alone: the leader line runs back to the point and would always
        # "collide" with the point's own marker.
        label.update_positions(renderer)
        box = Text.get_window_extent(label, renderer).expanded(1.02, 1.25)
        inside = plot_area.contains(box.x0, box.y0) and plot_area.contains(box.x1, box.y1)
        if inside and not any(box.overlaps(o) for o in occupied):
            occupied.append(box)
            break
        label.remove()
    else:  # nothing free: fall back to the default position
        ax.annotate(text, (row.pc1, row.pc2), xytext=CANDIDATES[0], va="bottom", **style)

plt.show()

In [ ]:
summary = (
    frame[frame.group != "anchor"]
    .groupby("group")["similarity to anchor"]
    .agg(["mean", "min", "max"])
)
summary.style.format("{:.3f}")

**Reading the result:** the similar sentences sit close to the anchor in the plot, and
the English translation lands next to its Spanish original, which is the cross-lingual
behavior the tutor relies on. The table confirms it without the 2D approximation: every
similar sentence scores higher than every different one.

## 4. Storage: the Chroma vector store

[Chroma](https://docs.trychroma.com/) is an open-source vector database that runs locally
(stored in `data/processed/chroma/`, about 1.1 GB). `python -m spanish_tutor.ingest.index_tatoeba`
embedded all 261,405 translated Tatoeba sentences into one collection, `tatoeba`.

Design decisions:

- **One document per translated sentence.** The embedded text is the Spanish sentence.
  Untranslated sentences are skipped, because the learner needs the meaning.
- **Deterministic ids** (`tatoeba:<sentence id>`). Indexing skips ids already stored, so an
  interrupted run resumes and a re-run adds nothing.
- **Metadata travels with each vector:** the English translation, the author
  (attribution), the token count, and the sentence's vocabulary (its `(lemma, pos)` set from
  `lexicon.py`).
- **The vocabulary is stored as a string** (`"comer|VERB;casa|NOUN"`), because Chroma
  metadata values must be scalars. `;` separates items because multi-word lemmas
  contain spaces.
- **Cosine distance** for the index.
- **Model guard:** the collection records which embedding model built it, and
  `open_store()` refuses to open it with a different model. Vectors from different models
  live in different spaces, and mixing them would give plausible-looking garbage.

In [ ]:
from spanish_tutor.vectorstore import decode_vocab, open_store, search_sentences

store = open_store()
collection = store._collection
print(f"collection {collection.name!r}: {collection.count():,} sentences")
print("collection metadata (model guard):", collection.metadata)

One stored record: its text, metadata and vector. Re-embedding the same text gives the
same vector (similarity 1.000), which confirms the store holds exactly what the model
produces:

In [ ]:
stored = collection.get(ids=["tatoeba:2503"], include=["documents", "metadatas", "embeddings"])
if not stored["ids"]:  # fall back to any stored record
    stored = collection.get(limit=1, include=["documents", "metadatas", "embeddings"])

text, meta, stored_vector = (
    stored["documents"][0],
    stored["metadatas"][0],
    np.array(stored["embeddings"][0]),
)
print("id:      ", stored["ids"][0])
print("text:    ", text)
for key, value in meta.items():
    print(f"{key + ':':9} {value}")
print("vocab →  ", sorted(decode_vocab(meta["vocab"])))
print(f"\nvector: {stored_vector.shape}, first 5: {stored_vector[:5].round(4)}")
print(
    f"similarity to a fresh embedding of the same text: {stored_vector @ np.array(embeddings.embed_query(text)):.3f}"
)

## 5. Lookups

### 5.1 Vocabulary-constrained sentence search

This is the core retrieval step the conversation skill will use. `search_sentences()`:

1. Embeds the query and fetches the 50 most similar sentences from Chroma.
2. Compares each sentence's vocabulary with your word bank (recognition mode).
3. Keeps sentences with at most `max_unknown` words you don't know: 0 means fully readable,
   1 means one new word to teach (Krashen's *i+1*).
4. Returns up to `k` in similarity order, listing any new words.

**Design decision:** the vocabulary filter runs in Python after the similarity search.
That's simple and fast enough for Phase 1. Phase 3's SQL difficulty index is its
long-term home.

First, the words you know, read from SQLite:

In [ ]:
known = db.known_vocabulary(
    conn
)  # SELECT lemma, pos FROM word_bank JOIN lexemes WHERE mode = 'recognition'
print(f"{len(known):,} words you recognize, e.g. {sorted(known)[:6]}")

Without the filter, plain similarity search returns sentences with words you haven't
learned. The `unknown words` column shows what the filter would remove:

In [ ]:
query = "I'm hungry"
raw_hits = store.similarity_search_with_score(query, k=10)
pd.DataFrame(
    [
        (
            1 - distance,
            doc.page_content,
            doc.metadata["en"],
            ", ".join(f"{l}|{p}" for l, p in sorted(decode_vocab(doc.metadata["vocab"]) - known)),
        )
        for doc, distance in raw_hits
    ],
    columns=["similarity", "Spanish", "English", "unknown words"],
).style.format({"similarity": "{:.3f}"})

In [ ]:
def show(hits):
    return pd.DataFrame(
        [
            (
                h.similarity,
                h.es,
                h.en,
                ", ".join(f"{l}|{p}" for l, p in sorted(h.unknown)),
                h.author,
            )
            for h in hits
        ],
        columns=["similarity", "Spanish", "English", "new words", "author"],
    ).style.format({"similarity": "{:.3f}"})


print(f"query {query!r}, only words you know:")
show(search_sentences(store, query, known, k=5, max_unknown=0))

In [ ]:
query = "¿Dónde está la estación?"
print(f"query {query!r}, one new word allowed (i+1):")
show(search_sentences(store, query, known, k=5, max_unknown=1))

### 5.2 Looking up words in the word bank

For each word, this query shows everything stored about it (definition, attributed
example, and whether you recognize and produce it). The query starts from the list of
requested words and `LEFT JOIN`s the tables, so a word that isn't stored at all, like
*estación* from the search above, still appears, with empty fields:

In [ ]:
pd.read_sql(
    """
    WITH requested(lemma) AS (VALUES ('hambre'), ('comer'), ('estación'))
    SELECT r.lemma, l.pos, l.definition_en, l.example_es, l.example_en,
           l.example_source, l.example_author,
           MAX(CASE WHEN b.mode = 'recognition' THEN 'yes' ELSE 'no' END) AS recognize,
           MAX(CASE WHEN b.mode = 'production'  THEN 'yes' ELSE 'no' END) AS produce,
           MIN(b.learned_at) AS learned_at
    FROM requested AS r
    LEFT JOIN lexemes   AS l ON l.lemma = r.lemma
    LEFT JOIN word_bank AS b ON b.lexeme_id = l.lexeme_id
    GROUP BY r.lemma, l.lexeme_id
    """,
    conn,
).set_index("lemma").T

## Summary

| Stage | What happens | Code |
|---|---|---|
| Download | Three sources into `data/raw/`; code is committed, data never is | `ingest/download.py` |
| Parse | SUBTLEX's three-block sheet, Tatoeba's TSVs joined with authors, compact Wiktionary entries | `ingest/subtlex.py`, `ingest/tatoeba.py`, `ingest/wiktionary.py` |
| Lemmatize | spaCy plus project rules turn text into `(lemma, pos)`; one shared path | `lexicon.py` |
| Frequency | Form counts split by real usage, filtered by Wiktionary | `seed.py` |
| Word bank | Event log plus trigger-maintained table in SQLite, recognition and production separate | `sql/schema.sql`, `db.py` |
| Vectorize | jina-embeddings-v2-base-es, 768-d unit vectors, CPU | `embeddings.py` |
| Store | Chroma collection with vocabulary metadata and a model guard | `vectorstore.py`, `ingest/index_tatoeba.py` |
| Look up | Similarity search filtered to words you know (or i+1) | `vectorstore.search_sentences` |

In [ ]:
conn.close()